# Building the candidate game list

Screens the Steam catalogue for multiplayer games with an early-access past and writes the candidate list the collector works from.

# Setup

In [6]:
import requests
import json
import re
import time
import random
import threading
import datetime as dt
from concurrent.futures import ThreadPoolExecutor,as_completed
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
catalogue_dir=project_root/'data'/'processed'/'catalogue'
meta_dir=project_root/'data'/'raw'/'meta'
for d in (catalogue_dir,meta_dir):
    d.mkdir(parents=True,exist_ok=True)

request_headers={'User-Agent':'EA-Health-Scoring-research/1.0 (academic study; mastudsaurabh33@gmail.com)'}
thread_local_storage=threading.local()

def get_session():
    if not hasattr(thread_local_storage,'session'):
        thread_local_storage.session=requests.Session()
        thread_local_storage.session.headers.update(request_headers)
    return thread_local_storage.session

# my inclusion criteria, all in one place
minimum_early_access_reviews=200
minimum_months_since_release=6
screening_workers=8
seconds_between_requests=0.4

print('project root:',project_root)
print('candidate list will be written to:',catalogue_dir/'candidate_games.csv')

project root: d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis
candidate list will be written to: d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\processed\catalogue\candidate_games.csv


# Finding every multiplayer game on steam

Walks the store search results, plus the early-access tag for games that have not launched yet.

In [7]:
def search_store(extra_params,label,max_pages=400):
    # walk the steam store search results and collect app ids
    found={}
    session=get_session()
    consecutive_blanks=0

    for page in range(max_pages):
        params=dict(query='',start=page*100,count=100,dynamic_data='',
                    sort_by='Released_DESC',category1=998,infinite=1,json=1,
                    ignore_preferences=1)
        params.update(extra_params)

        payload=None
        # a throttled request returns 200 with a null body
        for attempt in range(4):
            try:
                response=session.get('https://store.steampowered.com/search/results/',
                                     params=params,timeout=30)
                parsed=response.json()
                if isinstance(parsed,dict):
                    payload=parsed
                    break
            except Exception:
                pass
            time.sleep((2**attempt)+random.random())

        if payload is None:
            consecutive_blanks+=1
            print(f'  {label} page {page}: no usable response after retries '
                  f'({consecutive_blanks} in a row)',flush=True)
            # three dead pages means the end of the results
            if consecutive_blanks>=3:
                print(f'  {label}: stopping, end of results')
                break
            continue

        consecutive_blanks=0
        html=payload.get('results_html','')
        appids=re.findall(r'data-ds-appid="(\d+)"',html)
        names=re.findall(r'<span class="title">([^<]*)</span>',html)

        if not appids:
            print(f'  {label}: page {page} had no games, stopping')
            break
        for i,appid in enumerate(appids):
            # bundles carry several ids, i only want single games
            if ',' in appid:
                continue
            found.setdefault(int(appid),names[i] if i<len(names) else '')

        if page%20==0:
            print(f'  {label} page {page}: {len(found):,} games so far',flush=True)
        time.sleep(seconds_between_requests)
    return found


print('every multiplayer game')
multiplayer_games=search_store({'category2':1},'multiplayer')
print(f'multiplayer games found: {len(multiplayer_games):,}')

print('multiplayer games currently in early access')
# tag 493 is early access - the survivorship cohort
current_early_access=search_store({'category2':1,'tags':493},'early access')
print(f'currently in early access: {len(current_early_access):,}')

print(f'total multiplayer games      : {len(multiplayer_games):,}')
print(f'of those, still in early access: {len(current_early_access):,}')
print(f'launched or never in EA        : {len(multiplayer_games)-len(current_early_access):,}')

every multiplayer game
  multiplayer page 0: 100 games so far
  multiplayer page 20: 2,097 games so far
  multiplayer page 30: no usable response after retries (1 in a row)
  multiplayer page 40: 3,997 games so far
  multiplayer page 60: 5,996 games so far
  multiplayer page 61: no usable response after retries (1 in a row)
  multiplayer page 80: 7,894 games so far
  multiplayer page 92: no usable response after retries (1 in a row)
  multiplayer page 100: 9,793 games so far
  multiplayer page 120: 11,790 games so far
  multiplayer page 123: no usable response after retries (1 in a row)
  multiplayer page 140: 13,689 games so far
  multiplayer page 154: no usable response after retries (1 in a row)
  multiplayer page 160: 15,581 games so far
  multiplayer page 180: 17,580 games so far
  multiplayer page 185: no usable response after retries (1 in a row)
  multiplayer page 200: 19,477 games so far
  multiplayer page 216: no usable response after retries (1 in a row)
  multiplayer page 2

# Testing each game for an early access past

Probes each game's reviews for the early-access flag, using as few requests as possible.

In [8]:
screen_counter={'checked':0,'requests':0}
screen_lock=threading.Lock()

def fetch_reviews(appid,start_date=None,end_date=None,max_attempts=4):
    params={'json':1,'language':'all','num_per_page':100,'purchase_type':'all',
            'review_type':'all','filter':'recent','cursor':'*'}
    if start_date is not None:
        params.update({'filter':'all','start_date':int(start_date),
                       'end_date':int(end_date),'date_range_type':'include'})
    for attempt in range(max_attempts):
        try:
            with screen_lock:
                screen_counter['requests']+=1
            response=get_session().get(f'https://store.steampowered.com/appreviews/{appid}',
                                       params=params,timeout=30)
            if response.status_code==429 or response.status_code>=500:
                time.sleep((2**attempt)+random.random())
                continue
            payload=response.json()
            time.sleep(seconds_between_requests)
            return payload if payload.get('success')==1 else None
        except (requests.RequestException,ValueError):
            time.sleep((2**attempt)+random.random())
    return None


def screen_game(appid,name,is_currently_early_access):
    # decide inclusion in as few requests as possible
    result={'appid':appid,'name':name,'still_in_ea':is_currently_early_access,
            'total_reviews':0,'listing_date':None,'had_early_access':None,
            'ea_flags_in_probe':0,'verdict':'','months_since_listing':None}

    first_page=fetch_reviews(appid)
    with screen_lock:
        screen_counter['checked']+=1
    if first_page is None or not first_page.get('reviews'):
        result['verdict']='no_reviews'
        return result

    summary=first_page['query_summary']
    result['total_reviews']=summary.get('total_reviews',0)

    raw_listing=first_page['reviews'][0].get('app_release_date')
    listing_timestamp=int(float(raw_listing)) if raw_listing else None
    if listing_timestamp:
        result['listing_date']=dt.datetime.utcfromtimestamp(listing_timestamp).date().isoformat()
        result['months_since_listing']=round((time.time()-listing_timestamp)/(86400*30.4),1)

    # cheapest rejection first: under 200 total can never reach 200 early access
    if result['total_reviews']<minimum_early_access_reviews:
        result['verdict']='too_few_reviews'
        return result

    if not listing_timestamp:
        result['verdict']='no_listing_date'
        return result

    # still in early access, so survivorship cohort
    if is_currently_early_access:
        result['had_early_access']=True
        result['verdict']='candidate_still_in_ea'
        return result

    if result['months_since_listing'] is not None and result['months_since_listing']<minimum_months_since_release:
        result['verdict']='too_recent'
        return result

    # the only record of an early access past is the flag on old reviews
    # the listing date is unreliable in both directions,
    # for others before, so probing one direction would lose half the frame
    # probe both sides; one flag anywhere proves an early access past
    ea_flags=0
    probe_windows=[(listing_timestamp,listing_timestamp+90*86400),          # listing = EA start
                   (listing_timestamp-365*86400,listing_timestamp)]         # listing = launch
    for probe_start,probe_end in probe_windows:
        probe=fetch_reviews(appid,probe_start,probe_end)
        rows=probe.get('reviews',[]) if probe else []
        ea_flags+=sum(1 for r in rows if r.get('written_during_early_access'))
        if ea_flags>0:
            break

    result['ea_flags_in_probe']=ea_flags
    result['had_early_access']=ea_flags>0
    result['verdict']='candidate_launched' if ea_flags>0 else 'never_in_early_access'
    return result


print('screening every multiplayer game for an early access past')

to_screen=[(appid,name,appid in current_early_access) for appid,name in multiplayer_games.items()]
print(f'games to screen: {len(to_screen):,}  (up to 3 requests each)')
print(f'workers: {screening_workers}')

screen_results=[]
screen_started=time.time()

with ThreadPoolExecutor(max_workers=screening_workers) as pool:
    submitted={pool.submit(screen_game,a,n,ea):a for a,n,ea in to_screen}
    for done_count,finished in enumerate(as_completed(submitted),1):
        try:
            screen_results.append(finished.result())
        except Exception as problem:
            print(f'  app {submitted[finished]} screen failed: {type(problem).__name__}')
        if done_count%500==0:
            elapsed=time.time()-screen_started
            rate=done_count/max(elapsed,1)
            remaining=(len(to_screen)-done_count)/max(rate,0.001)/60
            print(f'  {done_count:,}/{len(to_screen):,} screened  '
                  f'{screen_counter["requests"]:,} requests  '
                  f'~{remaining:.0f} min left',flush=True)

print(f'screening finished in {(time.time()-screen_started)/60:.1f} minutes, '
      f'{screen_counter["requests"]:,} requests')

screening every multiplayer game for an early access past
games to screen: 21,330  (up to 3 requests each)
workers: 8
  500/21,330 screened  516 requests  ~31 min left
  1,000/21,330 screened  1,024 requests  ~31 min left
  1,500/21,330 screened  1,533 requests  ~29 min left
  2,000/21,330 screened  2,039 requests  ~29 min left
  2,500/21,330 screened  2,571 requests  ~28 min left
  3,000/21,330 screened  3,178 requests  ~28 min left
  3,500/21,330 screened  3,858 requests  ~29 min left
  4,000/21,330 screened  4,498 requests  ~28 min left
  4,500/21,330 screened  5,137 requests  ~28 min left
  5,000/21,330 screened  5,797 requests  ~28 min left
  5,500/21,330 screened  6,419 requests  ~27 min left
  6,000/21,330 screened  7,032 requests  ~26 min left
  6,500/21,330 screened  7,656 requests  ~25 min left
  7,000/21,330 screened  8,357 requests  ~25 min left
  7,500/21,330 screened  9,011 requests  ~24 min left
  8,000/21,330 screened  9,672 requests  ~24 min left
  8,500/21,330 screene

# Writing the candidate list

Applies the inclusion criteria and writes both the candidate list and the full screening record.

In [9]:
screening_frame=pd.DataFrame(screen_results)

print('WHAT THE SCREEN FOUND')
print(screening_frame['verdict'].value_counts().to_string())

candidates=screening_frame[screening_frame['verdict'].isin(['candidate_launched','candidate_still_in_ea'])].copy()
candidates['cohort']=candidates['verdict'].map({'candidate_launched':'launched',
                                                'candidate_still_in_ea':'still_in_ea'})
candidates=candidates.sort_values('total_reviews',ascending=False)

print(f'candidates for collection : {len(candidates):,}')
print(candidates['cohort'].value_counts().to_string())
print(f'total reviews across candidates : {candidates["total_reviews"].sum():,}')

# how the collector will route them, which drives the time estimate
whole_history_cursor_threshold=8000
small=candidates[candidates['total_reviews']<=whole_history_cursor_threshold]
large=candidates[candidates['total_reviews']>whole_history_cursor_threshold]
print(f'\nsmall enough to walk with the cursor (<= {whole_history_cursor_threshold:,} reviews): '
      f'{len(small):,} games')
print(f'large enough to need time windows: {len(large):,} games')

# cursor games cost total/100 requests, window games roughly the cap over 35
estimated_requests=(small['total_reviews']/100).sum()+len(large)*2*(5000/35)
print(f'\nrough request estimate for the sweep: {estimated_requests:,.0f}')
print(f'at 8 workers and ~1 request per second each: '
      f'{estimated_requests/8/3600:.1f} hours')

output_file_path=catalogue_dir/'candidate_games.csv'
candidates.to_csv(output_file_path,index=False)
print(f'\ncandidate list saved to {output_file_path}')

# the rejections are evidence for the sample section, so keep them
screening_frame.to_csv(catalogue_dir/'screening_full.csv',index=False)
print(f'full screening record saved to {catalogue_dir/"screening_full.csv"}')

print('TOP 15 CANDIDATES BY REVIEW COUNT')
print(candidates[['appid','name','cohort','total_reviews','listing_date']].head(15).to_string(index=False))

WHAT THE SCREEN FOUND
verdict
too_few_reviews          14585
never_in_early_access     3618
candidate_launched        1220
no_reviews                1018
candidate_still_in_ea      667
too_recent                 214
no_listing_date              8
candidates for collection : 1,887
cohort
launched       1220
still_in_ea     667
total reviews across candidates : 27,747,533

small enough to walk with the cursor (<= 8,000 reviews): 1,513 games
large enough to need time windows: 374 games

rough request estimate for the sweep: 129,217
at 8 workers and ~1 request per second each: 4.5 hours

candidate list saved to d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\processed\catalogue\candidate_games.csv
full screening record saved to d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\processed\catalogue\screening_full.csv
TOP 15 CANDIDATES BY REVIEW COUNT
  appid                  name      cohort  total_reviews listing_date
 252490                  Rust    launched

# The assertion cell

In [10]:
# if any of these fail the candidate list is not safe to collect from
print('CHECKS')

assert len(candidates)>0, 'no candidates at all, the screen is broken'
assert candidates['appid'].is_unique, 'duplicate app ids in the candidate list'
assert (candidates['total_reviews']>=minimum_early_access_reviews).all(), \
    'a candidate slipped through under the review threshold'
assert candidates['cohort'].isin(['launched','still_in_ea']).all(), 'unexpected cohort label'

launched_only=candidates[candidates['cohort']=='launched']
assert (launched_only['months_since_listing']>=minimum_months_since_release).all(), \
    'a launched candidate is more recent than my six month rule allows'

print(f'candidates            : {len(candidates):,}')
print(f'  launched            : {(candidates["cohort"]=="launched").sum():,}')
print(f'  still in early access: {(candidates["cohort"]=="still_in_ea").sum():,}')
print(f'app ids unique        : yes')
print(f'all above {minimum_early_access_reviews} reviews : yes')
print(f'all launched games >= {minimum_months_since_release} months old : yes')
print('\nALL CHECKS PASS - candidate_games.csv is ready for review_collection')

CHECKS
candidates            : 1,887
  launched            : 1,220
  still in early access: 667
app ids unique        : yes
all above 200 reviews : yes
all launched games >= 6 months old : yes

ALL CHECKS PASS - candidate_games.csv is ready for review_collection
